<a href="https://colab.research.google.com/github/leon72305-source/DSAI_FINAL/blob/main/DSandAIP2027_Drafts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## *Datasets Description*

| Column | Meaning |
| ----------- | ----------- |
| `df_chargers` | Dataset for Public Charging Infrastructure Statistics |
| `df_piv` | Dataset for Licensed Plug-In Vehicles (PIVs) Statistics|

## Environment Setup

The `-q` flag keeps the output quiet.

In [ ]:
!pip install -q odfpy

In [34]:
from google.colab import drive

drive.mount("/content/drive/")

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [35]:
import pandas as pd
import numpy as np

import re

import matplotlib.pyplot as plt
import seaborn as sns

## Loading the datasets

In [36]:
folder_path = "/content/drive/MyDrive/DataSet/"

df_piv = pd.read_csv(folder_path + "veh0142.csv", header=4)
df_chargers = pd.read_csv(folder_path + "evci9001.csv", header=2)

print("df_piv shape:", df_piv.shape)
print("df_chargers shape:", df_chargers.shape)

display(df_piv.head(3))
display(df_chargers.head(3))

df_piv shape: (26156, 74)
df_chargers shape: (470, 30)


,Units,BodyType,Fuel,Keepership,ONS Sort,ONS Code,ONS Geography,2026 Q2,2026 Q1,2025 Q4,...,2012 Q1,2011 Q4,2011 Q3,2011 Q2,2011 Q1,2010 Q4,2010 Q3,2010 Q2,2010 Q1,2009 Q4
0,Number,Buses and coaches,Battery electric,Company,1,K02000001,United Kingdom,"8,817","8,235","7,669",...,[x],[x],[x],[x],[x],[x],[x],[x],[x],[x]
1,Number,Buses and coaches,Battery electric,Company,2,K03000001,Great Britain,"8,549","7,988","7,425",...,54,58,57,59,58,49,45,41,40,38
2,Number,Buses and coaches,Battery electric,Company,3,E92000001,England,"7,333","6,804","6,276",...,51,55,54,57,55,46,43,39,38,36


,Local authority / region code,Local authority / region name,Oct-19,Jan-20,Apr-20,Jul-20,Oct-20,Jan-21,Apr-21,Jul-21,...,Apr-24,Jul-24,Oct-24 [Note 14],Jan-25,Apr-25,Jul-25,Oct-25,Jan-26,Apr-26,Jul-26
0,K02000001,United Kingdom,"15,116","16,505","17,947","18,265","19,487","20,775","22,790","24,374",...,"59,670","64,632","70,042","73,334","76,507","82,002","86,021","87,796","92,141","94,595"
1,K03000001,Great Britain,"14,821","16,210","17,642","17,953","19,169","20,455","22,463","24,044",...,"59,125","64,014","69,403","72,654","75,835","81,318","85,283","87,069","91,414","93,864"
2,E92000001,England,"12,549","13,719","14,979","15,395","16,456","17,459","19,261","20,563",...,"51,506","55,631","60,364","63,389","65,877","70,672","74,115","75,818","79,198","81,231"


In [40]:
def filter_veh(df):
    cols_to_keep = []
    for col in df.columns:
        col_str = str(col).strip()
        match = re.search(r'(20\d{2})\s*Q?([1-4])?', col_str)
        if match:
            year = int(match.group(1))
            q_match = match.group(2)
            quarter = int(q_match) if q_match else None

            # Keep if year is 2020 or later
            if year > 2019:
                cols_to_keep.append(col)
            # If exactly 2019, keep only Q4 onward (drop Q1, Q2, Q3)
            elif year == 2019:
                if quarter is None or quarter >= 4:
                    cols_to_keep.append(col)
        else:
            # Keep non-time metadata columns (like GSS code, LA name)
            cols_to_keep.append(col)
    return df[cols_to_keep]

def filter_chargers(df):
    cols_to_keep = []
    for col in df.columns:
        col_str = str(col).strip().lower()
        match = re.search(r'([a-z]{3})[-/\s]?(\d{2,4})', col_str)
        if match:
            month_str = match.group(1)
            year_raw = match.group(2)

            if month_str in month_map:
                month = month_map[month_str]
                year = int(year_raw)
                if year < 100:  # Convert 2-digit year (e.g., '19') to 2019
                    year += 2000

                # Rule: Keep up to April 2026
                if (year < 2026) or (year == 2026 and month <= 4):
                    cols_to_keep.append(col)
            else:
                cols_to_keep.append(col)
        else:
            # Keep non-time metadata columns
            cols_to_keep.append(col)
    return df[cols_to_keep]

df_piv = filter_veh(df_piv)
df_chargers = filter_chargers(df_chargers)

print("\n--- After Purging ---")
print(f"df_piv filtered shape: {df_piv.shape}")
print(f"df_chargers filtered shape: {df_chargers.shape}")

display(df_piv.head(3))
display(df_chargers.head(3))


--- After Purging ---
df_piv filtered shape: (26156, 34)
df_chargers filtered shape: (470, 29)


,Units,BodyType,Fuel,Keepership,ONS Sort,ONS Code,ONS Geography,2026 Q2,2026 Q1,2025 Q4,...,2022 Q1,2021 Q4,2021 Q3,2021 Q2,2021 Q1,2020 Q4,2020 Q3,2020 Q2,2020 Q1,2019 Q4
0,Number,Buses and coaches,Battery electric,Company,1,K02000001,United Kingdom,"8,817","8,235","7,669",...,"1,519","1,241","1,025",933,871,786,715,661,630,482
1,Number,Buses and coaches,Battery electric,Company,2,K03000001,Great Britain,"8,549","7,988","7,425",...,"1,481","1,237","1,023",933,871,786,715,661,630,482
2,Number,Buses and coaches,Battery electric,Company,3,E92000001,England,"7,333","6,804","6,276",...,"1,171","1,044",929,871,812,734,670,634,599,452


,Local authority / region code,Local authority / region name,Oct-19,Jan-20,Apr-20,Jul-20,Oct-20,Jan-21,Apr-21,Jul-21,...,Jan-24,Apr-24,Jul-24,Oct-24 [Note 14],Jan-25,Apr-25,Jul-25,Oct-25,Jan-26,Apr-26
0,K02000001,United Kingdom,"15,116","16,505","17,947","18,265","19,487","20,775","22,790","24,374",...,"53,677","59,670","64,632","70,042","73,334","76,507","82,002","86,021","87,796","92,141"
1,K03000001,Great Britain,"14,821","16,210","17,642","17,953","19,169","20,455","22,463","24,044",...,"53,212","59,125","64,014","69,403","72,654","75,835","81,318","85,283","87,069","91,414"
2,E92000001,England,"12,549","13,719","14,979","15,395","16,456","17,459","19,261","20,563",...,"46,374","51,506","55,631","60,364","63,389","65,877","70,672","74,115","75,818","79,198"
